# curebench results
- dataset: curebench_valset_pharse1.jsonl

In [1]:
import os
import sys
parent_dir = os.path.split(os.getcwd())[0]
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
print(f"Adding directory\n{parent_dir}\nto sys.path")

Adding directory
/Users/christiembp/Documents/workspace/CUREBench
to sys.path


In [2]:
import os
import json
import pandas as pd
import glob
from torch.utils.data import DataLoader
from dataset_utils import build_dataset

### load val dataset

In [3]:
def load_dataset_by_config(config_path):
  # load config file to get dataset info
  config = json.load(open(config_path, 'r')) if config_path else {}
  if 'dataset' in config:
      dataset_config = config['dataset']
      dataset_name = dataset_config.get('dataset_name', 'treatment')
  print(f"\nconfig file: {config_path}\ncontents:\n{dataset_config}")
  dataset_path = dataset_config.get("dataset_path")

  # build dataset
  dataset = build_dataset(
        dataset_config.get("dataset_path"),
    )
  dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
  dataset_list = []

  for batch in dataloader:
      question_type = batch[0][0]

      if question_type == "multi_choice":
          dataset_list.append({
              "question_type": batch[0][0],
              "id": batch[1][0],
              "question": batch[2][0],
              "answer": batch[3][0],
          })
      elif question_type == "open_ended_multi_choice":
          dataset_list.append({
              "question_type": batch[0][0],
              "id": batch[1][0],
              "question": batch[2][0],
              "answer": batch[3][0],
              "meta_question": batch[4][0],
          })
      elif question_type == "open_ended":
          dataset_list.append({
              "question_type": batch[0][0],
              "id": batch[1][0],
              "question": batch[2][0],
              "answer": batch[3][0],
          })
  return dataset_list

In [4]:
val_data_config_path= os.path.join(parent_dir, "metadata_config_val.json")
val_data_list = load_dataset_by_config(val_data_config_path)


config file: /Users/christiembp/Documents/workspace/CUREBench/metadata_config_val.json
contents:
{'dataset_name': 'cure_bench_phase1_val', 'dataset_path': '../resources/curebench_valset_pharse1.jsonl', 'description': 'CureBench 2025 val questions'}
dataset_path: ../resources/curebench_valset_pharse1.jsonl
CureBenchDataset initialized with 459 examples


In [5]:
# pandas settings
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [6]:
val_df = pd.DataFrame(val_data_list)

In [7]:
print(f"number of rows in val set: {len(val_df):,}")

number of rows in val set: 459


In [8]:
val_df.head()

,question_type,id,question,answer,meta_question
0,multi_choice,U9PHZ83RKYV8,Which drug brand name is associated with the treatment of acne?\nA: Salicylic Acid\nB: Minoxidil\nC: Ketoconazole\nD: Fluocinonide,A,NaN
1,open_ended_multi_choice,vIGwm8qguXYi,What should patients do if they experience severe allergic reactions during or after receiving fosaprepitant for injection?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: What should patients do if they experience severe allergic reactions during or after receiving fosaprepitant for injection?\nA: Wait for the symptoms to resolve on their own.\nB: Inform their healthcare provider immediately and seek emergency medical care.\nC: Stop chemotherapy treatment permanently.\nD: Take over-the-counter antihistamines.\n\n"
2,open_ended_multi_choice,GlpDnJvMaWbs,What should you do if the dose indicator on Stiolto Respimat reaches 0?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: What should you do if the dose indicator on Stiolto Respimat reaches 0?\nA: Continue using the inhaler until the cartridge is empty.\nB: Prepare and use a new Stiolto Respimat inhaler.\nC: Turn the clear base to reset the dose indicator.\nD: Clean the mouthpiece and continue using the inhaler.\n\n"
3,open_ended_multi_choice,WfWiWK0yULaX,Which of the following conditions is a contraindication for the use of Gadavist?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: Which of the following conditions is a contraindication for the use of Gadavist?\nA: Mild hypersensitivity reactions to Gadavist\nB: History of severe hypersensitivity reactions to Gadavist\nC: Renal impairment\nD: Liver dysfunction\n\n"
4,multi_choice,wzkMQ7uHtlLs,"What is the primary consideration for lactating mothers using Albuterol Sulfate HFA?\nA: It is contraindicated during lactation.\nB: Plasma levels of albuterol are low, and effects on breastfed children are likely minimal.\nC: It significantly reduces milk production.\nD: It should only be used in emergencies.",B,NaN


In [9]:
val_df["question_index"] = val_df.index

## read in partial results and stich them together

In [10]:
results_path = os.path.join(parent_dir, "resources/curebench-val-results")

In [11]:
file_pattern = os.path.join(results_path, '*.csv')

In [12]:
all_files = glob.glob(file_pattern)

In [13]:
df_list = [pd.read_csv(f) for f in all_files]

In [14]:
results_df = pd.concat(df_list, ignore_index=True)

In [15]:
print(f"number of observations in combined results: {len(results_df)}")

number of observations in combined results: 565


In [16]:
all_files

['/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_2.csv',
 '/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_3.csv',
 '/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_4.csv',
 '/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_5.csv',
 '/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_6.csv']

In [17]:
for i, x in enumerate(df_list):
    print(all_files[i], len(x))

/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_2.csv 106
/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_3.csv 100
/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_4.csv 200
/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_5.csv 100
/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission_6.csv 59


In [18]:
results_df.drop_duplicates()
print(len(results_df))

565


In [19]:
results_df.head(2)

,id,prediction,choice,reasoning
0,U9PHZ83RKYV8,A,A,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The question is multiple-choice, asking for a drug brand name associated with acne treatment. We need to pick the correct option. Salicylic Acid is widely used for acne. So answer is A.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""A""}]"
1,vIGwm8qguXYi,"Patients should immediately seek emergency medical attention if they experience severe allergic reactions during or after receiving fosaprepitant, and inform their healthcare provider about the symptoms.",B,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""Need to provide response: It's an open-ended question. Provide succinct answer.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""Patients should immediately seek emergency medical attention if they experience severe allergic reactions during or after receiving fosaprepitant, and inform their healthcare provider about the symptoms.""}, {""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The question is multiple choice. The correct option is B.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""B""}]"


In [20]:
results_df = pd.merge(results_df, val_df, on='id', how='left')

In [21]:
results_df.head(2)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,question_index
0,U9PHZ83RKYV8,A,A,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The question is multiple-choice, asking for a drug brand name associated with acne treatment. We need to pick the correct option. Salicylic Acid is widely used for acne. So answer is A.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""A""}]",multi_choice,Which drug brand name is associated with the treatment of acne?\nA: Salicylic Acid\nB: Minoxidil\nC: Ketoconazole\nD: Fluocinonide,A,NaN,0
1,vIGwm8qguXYi,"Patients should immediately seek emergency medical attention if they experience severe allergic reactions during or after receiving fosaprepitant, and inform their healthcare provider about the symptoms.",B,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""Need to provide response: It's an open-ended question. Provide succinct answer.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""Patients should immediately seek emergency medical attention if they experience severe allergic reactions during or after receiving fosaprepitant, and inform their healthcare provider about the symptoms.""}, {""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The question is multiple choice. The correct option is B.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""B""}]",open_ended_multi_choice,What should patients do if they experience severe allergic reactions during or after receiving fosaprepitant for injection?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: What should patients do if they experience severe allergic reactions during or after receiving fosaprepitant for injection?\nA: Wait for the symptoms to resolve on their own.\nB: Inform their healthcare provider immediately and seek emergency medical care.\nC: Stop chemotherapy treatment permanently.\nD: Take over-the-counter antihistamines.\n\n",1


In [22]:
question_frequencies = results_df['question_index'].value_counts()

In [23]:
question_frequencies[question_frequencies > 1].head()

question_index
141    2
128    2
119    2
120    2
121    2
Name: count, dtype: int64

In [25]:
results_df[results_df.prediction=="Error"].head(2)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,question_index
7,dlKfnTKdPf9G,Error,NOTAVALUE,"""Error occurred during inference""",open_ended_multi_choice,What should a patient do if they experience severe allergic reactions while taking PERTZYE?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: What should a patient do if they experience severe allergic reactions while taking PERTZYE?\nA: Continue taking the medication and monitor symptoms\nB: Stop taking PERTZYE and seek emergency treatment immediately\nC: Reduce the dose and consult their doctor\nD: Take an over-the-counter antihistamine\n\n",7
8,rYhpGH3kQW8P,Error,NOTAVALUE,"""Error occurred during inference""",open_ended_multi_choice,What is the recommended action if a patient’s serum potassium level reaches 6.0 mEq/L while on Inspra therapy?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: What is the recommended action if a patient’s serum potassium level reaches 6.0 mEq/L while on Inspra therapy?\nA: Reduce the dose to 25 mg every other day\nB: Withhold Inspra therapy\nC: Continue the current dose\nD: Increase the dose to 50 mg once daily\n\n",8


#### drop errors and duplicates

In [26]:
non_error_df = results_df[results_df.prediction!="Error"].copy()

In [27]:
deduplicated_df = non_error_df.drop_duplicates(subset=['id'], keep='first')

In [28]:
len(deduplicated_df )

459

### compute results

In [29]:
deduplicated_df.head(1)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,question_index
0,U9PHZ83RKYV8,A,A,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The question is multiple-choice, asking for a drug brand name associated with acne treatment. We need to pick the correct option. Salicylic Acid is widely used for acne. So answer is A.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""A""}]",multi_choice,Which drug brand name is associated with the treatment of acne?\nA: Salicylic Acid\nB: Minoxidil\nC: Ketoconazole\nD: Fluocinonide,A,NaN,0


In [30]:
deduplicated_df["correct"] = (deduplicated_df['choice'] == deduplicated_df['answer'])

/var/folders/1n/sp9kxl4s5szfp36xy6p8_1kw0000gn/T/ipykernel_84014/3385194086.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  deduplicated_df["correct"] = (deduplicated_df['choice'] == deduplicated_df['answer'])


In [31]:
deduplicated_df.head(1)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,question_index,correct
0,U9PHZ83RKYV8,A,A,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The question is multiple-choice, asking for a drug brand name associated with acne treatment. We need to pick the correct option. Salicylic Acid is widely used for acne. So answer is A.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""A""}]",multi_choice,Which drug brand name is associated with the treatment of acne?\nA: Salicylic Acid\nB: Minoxidil\nC: Ketoconazole\nD: Fluocinonide,A,NaN,0,True


In [32]:
non_open_ended_df = deduplicated_df[deduplicated_df.question_type!="open_ended"]

In [33]:
len(non_open_ended_df[non_open_ended_df.correct==True])

269

In [34]:
# Calculate and log overall accuracy
total_correct = len(non_open_ended_df[non_open_ended_df.correct==True])
total_examples = len(non_open_ended_df)
overall_accuracy = total_correct / total_examples if total_examples > 0 else 0.0

print(f"Overall accuracy (excluding open-ended questions): {overall_accuracy:.2%} ({total_correct}/{total_examples})")

Overall accuracy (excluding open-ended questions): 65.13% (269/413)


### inspect output

In [37]:
correct_df = non_open_ended_df[non_open_ended_df.correct==True]

In [47]:
wrong_df = non_open_ended_df[non_open_ended_df.correct!=True]

In [48]:
wrong_df.head(1)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,question_index,correct
6,cRbl2ruD1WMZ,A,A,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""We need to determine which option is NOT an indicated use for \""up and up ibuprofen\"". Likely it's a brand name for ibuprofen. Indications: minor aches and pains, occasional sleeplessness? Actually ibuprofen is analgesic; not for sleeplessness; maybe \""up and up\"" is a brand? The question seems trick: they ask which usage is NOT indicated for ibuprofen. Options: A- Relief of occasional sleeplessness, B- Relief of minor aches and pains, C- Treatment of chronic pain conditions, D- Helping users stay asleep. The correct answer likely A: relief of occasional sleeplessness is NOT indicated. So answer: A.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""A""}]",multi_choice,Which of the following is NOT an indicated use for 'up and up ibuprofen'?\nA: Relief of occasional sleeplessness\nB: Relief of minor aches and pains\nC: Treatment of chronic pain conditions\nD: Helping users stay asleep,C,NaN,6,False


In [41]:
correct_question_counts = correct_df.groupby(['question_type'], as_index=False).size() # get counts
correct_question_counts.sort_values('size', ascending=False, inplace=True) # sort by size

In [42]:
correct_question_counts

,question_type,size
1,open_ended_multi_choice,135
0,multi_choice,134


In [49]:
wrong_question_counts = wrong_df.groupby(['question_type'], as_index=False).size() # get counts
wrong_question_counts.sort_values('size', ascending=False, inplace=True) # sort by size

In [50]:
wrong_question_counts

,question_type,size
1,open_ended_multi_choice,95
0,multi_choice,49


In [51]:
wrong_df[wrong_df.question_type=="open_ended_multi_choice"]

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,question_index,correct
16,sMLyoU0d3Mrn,"If blood pressure remains inadequately controlled in a patient taking NURTEC ODT, consider reviewing the dosage, ensuring adherence, evaluating for potential drug interactions, assessing lifestyle factors affecting blood pressure, and possibly adjusting the treatment plan, including adding or switching to other antihypertensive agents as appropriate.",D,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""User asks a medical question. Likely open-ended.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""If blood pressure remains inadequately controlled in a patient taking NURTEC ODT, consider reviewing the dosage, ensuring adherence, evaluating for potential drug interactions, assessing lifestyle factors affecting blood pressure, and possibly adjusting the treatment plan, including adding or switching to other antihypertensive agents as appropriate.""}, {""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The question is multiple choice (MC). The agent's answer provided is an open-ended response. The question states, \""The following is a multiple choice question about medicine and the agent's open-ended answer to the question...\"". The agent's answer is essentially a short summary of what should be considered, but it does not match the multiple-choice options directly. \n\nGiven the instructions to convert the open-ended answer to a final answer format with the corresponding option label, there are no direct options that match the exact content. The best approach is to determine how each option might align with the content:\n\nA: Increasing the dosage of NURTEC ODT\nB: Discontinuing NURTEC ODT if no alternative etiology is found\nC: Switching to another CGRP antagonist\nD: Prescribing additional antihypertensive medications\n\nGiven the answer mentions reviewing dosage, ensuring adherence, evaluating for potential drug interactions, assessing lifestyle factors, and possibly adjusting the treatment plan: \n- Option A might involve reviewing the dosage, which is part of the content of the answer.\n- Option B does not directly align with the content.\n- Option C might involve switching, which is part of the answer.\n- Option D involves possibly adding or switching antihypertensive agents.\n\nThe most applicable to the content of the answer is option D, as it directly addresses adding or adjusting the treatment plan, including adding or switching. Therefore, the best match is:\n\nAnswer: D.\n\nThis is based on the fact that the agent's answer emphasizes adjusting the treatment plan, which includes the idea of adding or switching antihypertensive agents.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""D""}]",open_ended_multi_choice,What should be considered if blood pressure remains inadequately controlled in a patient taking NURTEC ODT?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: What should be considered if blood pressure remains inadequately controlled in a patient taking NURTEC ODT?\nA: Increasing the dosage of NURTEC ODT\nB: Discontinuing NURTEC ODT if no alternative etiology is found\nC: Switching to another CGRP antagonist\nD: Prescribing additional antihypertensive medications\n\n",11,False
29,Eic83tab2ldk,Albuterol sulfate demonstrated no evidence of impaired fertility in rats at an oral dose of 300 mg/kg body weight.,NOTAVALUE,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""This is an open-ended question (OE) about the specific dose of albuterol sulfate that did not show impaired fertility in rats.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""Albuterol sulfate demonstrated no evi

In [52]:
wrong_df[wrong_df.question_type=="open_ended_multi_choice"].groupby(['choice'], as_index=False).size()

,choice,size
0,A,13
1,B,5
2,C,12
3,D,13
4,E,6
5,NOTAVALUE,46


In [53]:
task_types = [
  {
    "task_type": "Treatment Recommendation",
    "description": "Questions regarding specialized treatment recommendations considering patient populations."
  },
  {
    "task_type": "Adverse Event",
    "description": "Predicting potential adverse events and side effects based on drug properties and patient factors."
  },
  {
    "task_type": "Drug Overview",
    "description": "Package label principal display panel and comprehensive drug description."
  },
  {
    "task_type": "Drug Ingredients",
    "description": "Product data elements and active/inactive ingredient analysis."
  },
  {
    "task_type": "Drug Warnings and Safety",
    "description": "Boxed warnings, contraindications, adverse reactions, and drug interactions."
  },
  {
    "task_type": "Drug Dependence and Abuse",
    "description": "Drug abuse potential, dependence, controlled substance classification, and overdosage."
  },
  {
    "task_type": "Dosage and Administration",
    "description": "Indications, usage, administration, dosage forms, strengths, and instructions."
  },
  {
    "task_type": "Drug Use in Specific Populations",
    "description": "Usage in pregnancy, pediatric, geriatric populations, and nursing mothers."
  },
  {
    "task_type": "Pharmacology",
    "description": "Clinical pharmacology, mechanism of action, pharmacodynamics, and pharmacokinetics."
  },
  {
    "task_type": "Clinical Information",
    "description": "Clinical studies and trial data analysis."
  },
  {
    "task_type": "Nonclinical Toxicology",
    "description": "Toxicology, carcinogenesis, mutagenesis, fertility impairment, and animal studies."
  },
  {
    "task_type": "Patient-Focused Information",
    "description": "Patient medication guides, package inserts, and patient information."
  }
]

In [54]:
print(f"number of task types: {len(task_types)}")

number of task types: 12
